# TrustScoreAI appraisal NLP training workflow

This notebook rebuilds the two text models from the labeled appraisal workbook and evaluates them on held-out **template families**. It keeps the agreed division of work:

- **Competency model:** multi-label. One answer can map to zero, one, or several of Communication, Collaboration, Compliance Awareness, and Learning and Development.
- **Direction model:** predicts negative, positive, mixed, or neutral for a statement and competency pair. It is trained only on relevant annotated examples.
- **Scoring:** the deployed app uses the predicted direction for supervisor appraisal answers only. Positive and negative predictions contribute 100 and 0 respectively; mixed and neutral do not receive a numeric value. Employee self-questionnaire answers are context for the supervisor and are not classified or scored.

The existing `appraisal_nlp_baseline_colab.ipynb` is left untouched. This cleaned copy is designed to run from top to bottom in Google Colab.

In [ ]:
%pip -q install pandas openpyxl scikit-learn==1.9.0 scipy joblib


## 1. Upload the labeled workbook

Choose `appraisal_statement_annotation_2000_set_master_columns.xlsx` (or the current adjudicated workbook with the same columns). The loader searches all worksheets for the required fields, so the sheet name does not have to match a particular title.

In [ ]:
from google.colab import files
import hashlib
import pandas as pd

uploaded = files.upload()
workbook_path = next(iter(uploaded))
book = pd.ExcelFile(workbook_path)
required_columns = {
    "appraisal_statement", "category",
    "adjudicated_relevant", "adjudicated_direction"
}

data = None
sheet_used = None
for sheet_name in book.sheet_names:
    candidate = pd.read_excel(workbook_path, sheet_name=sheet_name)
    if required_columns.issubset(candidate.columns):
        data, sheet_used = candidate, sheet_name
        break
if data is None:
    raise ValueError(f"No worksheet has all required columns: {sorted(required_columns)}")

print("Worksheet:", sheet_used)
print("Rows and columns:", data.shape)
print("Columns:", data.columns.tolist())
display(data.head())

## 2. Validate labels and make one row per statement

The workbook is in long form: each statement normally has one row per competency. We check duplicate pairs before converting relevance labels into a four-column multi-label target. Conflicting labels must be corrected in the workbook rather than silently discarded.

In [ ]:
import numpy as np
import re

CATEGORIES = [
    "collaboration", "communication",
    "compliance_awareness", "learning_development"
]
DIRECTIONS = ["negative", "positive", "mixed", "neutral"]

data = data.copy()
has_text_group_id = "text_group_id" in data.columns
data["category"] = data["category"].astype(str).str.strip().str.lower()
data["appraisal_statement"] = data["appraisal_statement"].fillna("").astype(str).str.strip()
if has_text_group_id:
    data["text_group_id"] = data["text_group_id"].astype(str).str.strip()
else:
    def stable_text_id(value):
        normalized = re.sub(r"\s+", " ", value.casefold()).strip()
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:20]
    data["text_group_id"] = data["appraisal_statement"].map(stable_text_id)
data["relevant"] = data["adjudicated_relevant"].astype(str).str.strip().str.lower().map(
    {"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0}
)
data["direction"] = data["adjudicated_direction"].fillna("").astype(str).str.strip().str.lower()

if data["relevant"].isna().any():
    raise ValueError("Unexpected or missing adjudicated_relevant values; expected yes/no.")
if not set(data["category"]).issubset(CATEGORIES):
    raise ValueError("The workbook contains a category outside the four agreed competencies.")
if data["text_group_id"].eq("").any() or data["appraisal_statement"].eq("").any():
    raise ValueError("A statement ID or appraisal statement is blank.")

pair_conflicts = data.groupby(["text_group_id", "category"])["relevant"].nunique()
if (pair_conflicts > 1).any():
    raise ValueError(f"{int((pair_conflicts > 1).sum())} statement/category pairs have conflicting relevance labels.")
statement_conflicts = data.groupby("text_group_id")["appraisal_statement"].nunique()
if has_text_group_id and (statement_conflicts > 1).any():
    raise ValueError("A text_group_id is connected to more than one statement.")

data = data.drop_duplicates(["text_group_id", "category"], keep="first").copy()
labels = data.pivot(index="text_group_id", columns="category", values="relevant")
labels = labels.reindex(columns=CATEGORIES).fillna(0).astype(int)
statements = (
    data.drop_duplicates("text_group_id")
    .set_index("text_group_id")
    .loc[labels.index, "appraisal_statement"]
    .astype(str)
)

print("Unique statements:", len(statements))
print("Statement/category rows:", len(data))
print("Positive labels by competency:", labels.sum().to_dict())
print("Direction labels on relevant rows:")
display(data.loc[data["relevant"].eq(1), "direction"].value_counts(dropna=False))

## 3. Build template families to prevent near-duplicate leakage

Statements with very similar wording are connected into one family. The 0.92 cosine-similarity cutoff matches the duplicate audit in the original notebook. All members of a family stay in one partition.

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

family_frame = pd.DataFrame({
    "text_group_id": statements.index.astype(str),
    "appraisal_statement": statements.to_numpy(),
})
family_frame["normalized_text"] = family_frame["appraisal_statement"].str.lower().map(
    lambda value: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", value)).strip()
)
family_vectorizer = TfidfVectorizer(ngram_range=(1, 2), strip_accents="unicode", min_df=1)
family_matrix = family_vectorizer.fit_transform(family_frame["normalized_text"])
similarity_cutoff = 0.92
similarities = cosine_similarity(family_matrix)
connections = csr_matrix(similarities >= similarity_cutoff)
connections.setdiag(0)
connections.eliminate_zeros()
family_count, family_labels = connected_components(connections, directed=False)
family_frame["template_family_id"] = [f"F{value:04d}" for value in family_labels]
family_by_id = family_frame.set_index("text_group_id")["template_family_id"]
groups = family_by_id.reindex(labels.index.astype(str)).to_numpy()
if pd.isna(groups).any():
    raise ValueError("Some statements have no template family.")
print("Template families:", family_count)
print("Families with multiple statements:", int((family_frame["template_family_id"].value_counts() > 1).sum()))

## 4. Create grouped train, validation, and test partitions

The split is approximately 70% train, 15% validation, and 15% test. Thresholds are selected on validation data only. The test partition is held untouched until final evaluation.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = statements.reindex(labels.index).reset_index(drop=True)
Y = labels.reset_index(drop=True)
first = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(first.split(X, Y, groups=groups))
second = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=43)
train_relative_idx, val_relative_idx = next(second.split(
    X.iloc[train_val_idx], Y.iloc[train_val_idx], groups=groups[train_val_idx]
))
train_idx = train_val_idx[train_relative_idx]
val_idx = train_val_idx[val_relative_idx]

X_train, Y_train = X.iloc[train_idx], Y.iloc[train_idx]
X_val, Y_val = X.iloc[val_idx], Y.iloc[val_idx]
X_test, Y_test = X.iloc[test_idx], Y.iloc[test_idx]
train_families, val_families, test_families = (
    set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
)
assert train_families.isdisjoint(val_families | test_families)
assert val_families.isdisjoint(test_families)
for name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    print(f"{name}: {len(idx)} statements, {len(set(groups[idx]))} template families")
    print("  positive labels:", Y.iloc[idx].sum().to_dict())

## 5. Train and evaluate the multi-label competency model

Each competency gets an independent probability. A statement may pass the cutoff for several competencies. The TF-IDF + One-vs-Rest Logistic Regression settings match the notebook baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score


def make_competency_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=1, sublinear_tf=True, strip_accents="unicode"
        )),
        ("classifier", OneVsRestClassifier(LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42
        ))),
    ])

competency_model = make_competency_model()
competency_model.fit(X_train, Y_train)
validation_probabilities = competency_model.predict_proba(X_val)
threshold_grid = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
competency_thresholds = {}
for column, category in enumerate(CATEGORIES):
    actual = Y_val.iloc[:, column].to_numpy()
    candidates = [
        (f1_score(actual, validation_probabilities[:, column] >= cutoff, zero_division=0), cutoff)
        for cutoff in threshold_grid
    ]
    competency_thresholds[category] = max(candidates, key=lambda pair: (pair[0], pair[1]))[1]
print("Thresholds selected on validation:", competency_thresholds)

test_probabilities = competency_model.predict_proba(X_test)
test_predictions = (test_probabilities >= np.array([competency_thresholds[c] for c in CATEGORIES])).astype(int)
print("\nHeld-out competency test results:")
print(classification_report(Y_test, test_predictions, target_names=CATEGORIES, zero_division=0))

## 6. Train and evaluate direction for relevant answers

This model receives `competency [SEP] statement`. It sees only relevant rows with an adjudicated direction. Its test report includes mixed and neutral so the notebook shows when a prediction could receive a numeric score incorrectly.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

id_to_split = {}
for split_name, indexes in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    for index in indexes:
        id_to_split[str(labels.index[index])] = split_name

data["split"] = data["text_group_id"].astype(str).map(id_to_split)
data["template_family_id"] = data["text_group_id"].astype(str).map(family_by_id)
direction_rows = data[
    data["relevant"].eq(1) & data["direction"].isin(DIRECTIONS)
].copy()
direction_rows["model_text"] = (
    direction_rows["category"]
    + " [SEP] " + direction_rows["appraisal_statement"]
)
train_direction = direction_rows[direction_rows["split"] == "train"]
validation_direction = direction_rows[direction_rows["split"] == "validation"]
test_direction = direction_rows[direction_rows["split"] == "test"]

def make_direction_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=1, sublinear_tf=True, strip_accents="unicode"
        )),
        ("classifier", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42
        )),
    ])

direction_model = make_direction_model()
direction_model.fit(train_direction["model_text"], train_direction["direction"])
validation_direction_predictions = direction_model.predict(validation_direction["model_text"])
print("Validation direction results:")
print(classification_report(
    validation_direction["direction"], validation_direction_predictions,
    labels=DIRECTIONS, zero_division=0
))

In [ ]:
test_direction_predictions = direction_model.predict(test_direction["model_text"])
print("Held-out direction test results:")
print(classification_report(
    test_direction["direction"], test_direction_predictions,
    labels=DIRECTIONS, zero_division=0
))
print("Confusion matrix (actual rows, predicted columns):")
print(confusion_matrix(
    test_direction["direction"], test_direction_predictions,
    labels=DIRECTIONS
))
scored_prediction = np.isin(test_direction_predictions, ["positive", "negative"])
actual_uncertain = test_direction["direction"].isin(["mixed", "neutral"]).to_numpy()
print("\nPredicted numeric score coverage:", f"{scored_prediction.mean():.1%}")
print("Mixed/neutral examples predicted positive or negative:", int((scored_prediction & actual_uncertain).sum()))

## 7. Evaluate the combined pipeline and scoring policy

The competency output decides whether each statement is assigned to each category. Only assigned pairs receive a direction prediction. This combined report measures the end-to-end label: irrelevant, negative, positive, mixed, or neutral.

In [ ]:
# Multi-label probabilities for every statement in the held-out test set.
statement_test_probs = pd.DataFrame(test_probabilities, columns=CATEGORIES)
statement_test_probs["text_group_id"] = labels.index[test_idx].astype(str).to_numpy()
prob_lookup = statement_test_probs.set_index("text_group_id")
full_test = data[data["text_group_id"].astype(str).isin(prob_lookup.index)].copy()
full_test = full_test[full_test["relevant"].eq(0) | full_test["direction"].isin(DIRECTIONS)].copy()
full_test["relevance_probability"] = [
    float(prob_lookup.loc[row.text_group_id, row.category]) for row in full_test.itertuples()
]
full_test["predicted_relevant"] = [
    probability >= competency_thresholds[category]
    for probability, category in zip(full_test["relevance_probability"], full_test["category"])
]
full_test["model_text"] = (
    full_test["category"]
    + " [SEP] " + full_test["appraisal_statement"]
)
full_test["predicted_direction"] = direction_model.predict(full_test["model_text"])
full_test["actual_label"] = np.where(
    full_test["relevant"].eq(0), "irrelevant", full_test["direction"]
)
full_test["predicted_label"] = np.where(
    full_test["predicted_relevant"], full_test["predicted_direction"], "irrelevant"
)
E2E_LABELS = ["irrelevant", "negative", "positive", "mixed", "neutral"]
print("Combined held-out test results:")
print(classification_report(
    full_test["actual_label"], full_test["predicted_label"],
    labels=E2E_LABELS, zero_division=0
))

## 8. Refit deployable models and export artifacts

After evaluating the untouched test partition, the final artifacts are fitted on train plus validation. The test partition remains excluded. The thresholds stay fixed at the validation-selected values. Download the three exported files and place them beside the FastAPI `main.py` file:

- `trustscore_competency_model.joblib`
- `trustscore_competency_model_metadata.json`
- `trustscore_direction_model.joblib`
- `trustscore_direction_model_metadata.json`

The metadata keeps the measured test metrics separate from the final models fitted on development data.

In [ ]:
import json
import joblib

# Refit on train + validation only; keep the test set out of both artifacts.
dev_indexes = np.concatenate([train_idx, val_idx])
X_development = X.iloc[dev_indexes]
Y_development = Y.iloc[dev_indexes]
category_model_final = make_competency_model().fit(X_development, Y_development)
direction_development = direction_rows[direction_rows["split"].isin(["train", "validation"])]
direction_model_final = make_direction_model().fit(
    direction_development["model_text"], direction_development["direction"]
)

category_bundle = {
    "model": category_model_final,
    "categories": CATEGORIES,
    "thresholds": competency_thresholds,
    "model_version": "tfidf-ovr-appraisal-competencies-v1",
}
joblib.dump(category_bundle, "trustscore_competency_model.joblib")
category_metadata = {
    "task": "multi-label appraisal competency classification",
    "categories": CATEGORIES,
    "thresholds": competency_thresholds,
    "threshold_source": "validation split only",
    "model": "word TF-IDF (1,2) + One-vs-Rest Logistic Regression",
    "training_statements": int(len(X_development)),
    "grouping": "0.92 cosine-similarity template families; test families excluded",
    "statement_id_source": "workbook text_group_id" if has_text_group_id else "SHA-256 of normalized statement text",
}
with open("trustscore_competency_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(category_metadata, f, indent=2)

joblib.dump(direction_model_final, "trustscore_direction_model.joblib")
direction_classifier = direction_model_final.named_steps["classifier"]
direction_metadata = {
    "task": "appraisal direction classification",
    "class_order": [str(label) for label in direction_classifier.classes_],
    "input_format": "lowercase category + ' [SEP] ' + cleaned appraisal statement",
    "scoring_policy": {
        "positive": "automatic score contribution of 100",
        "negative": "automatic score contribution of 0",
        "mixed": "no numerical score",
        "neutral": "no numerical score",
        "human_confirmation_required": False,
    },
    "training_rows": int(len(direction_development)),
    "grouping": "same held-out template-family split as competency model",
}
with open("trustscore_direction_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(direction_metadata, f, indent=2)

print("Saved the competency and direction model artifacts.")
print("Thresholds:", competency_thresholds)

In [ ]:
files.download("trustscore_competency_model.joblib")
files.download("trustscore_competency_model_metadata.json")
files.download("trustscore_direction_model.joblib")
files.download("trustscore_direction_model_metadata.json")

## Recorded baseline from the previous run

The saved notebook run reported **0.97 macro F1** for the multi-label category model on the grouped held-out test set. The direction model reported **0.80 accuracy** and **0.72 macro F1** on its grouped held-out set. Its weakest results were on mixed and neutral examples. Re-run this rebuilt notebook to reproduce current results; the printed results, not these historical reference values, are the ones to report for a fresh run.

The deployed score is a transparent average of eight group scores; the Random Forest regressor is not used to assign competency labels or directions. This notebook trains the two text classifiers used before that score calculation.